# EM synapse mapping

Creating a synaptome from a morphology with spines.

# Authentication

Follow the instructions below to authenticate and select a project to work in.

The morphology-with-spines that is to serve as the input of `Synaptome` generation must be accessible from the project.

In [ ]:
import obi_auth

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import CellMorphology, EMCellMesh, EMDenseReconstructionDataset, Circuit
from entitysdk import Client
from entitysdk._server_schemas import AssetLabel

token = obi_auth.get_token(environment="staging", auth_mode="daf")
project_context = get_projects(token, env="staging")

# Find a skeletonized morphology as input

We search for morphologies originating from the `IARPA MICrONS mouse`, as they are derived by skeletonization

In [ ]:
from entitysdk.models import MEModel

entity_client = Client(token_manager=token, project_context=project_context, environment="staging")

entity_id_str = "FILL IN HERE!"

skeletonized_morphology = entity_client.get_entity(entity_type=MEModel, entity_id=entity_id_str)

# Set up task

Now create a CAVE token.

In [ ]:
import caveclient
temporary_client = caveclient.CAVEclient()
temporary_client.auth.get_new_token()

Your token needs to be pasted below.

In [ ]:
import os
os.environ['CAVECLIENT_MICRONS_API_KEY'] = "TOKEN HERE!"

In [ ]:
from obi_one.scientific.tasks.em_synapse_mapping.config import EMSynapseMappingSingleConfig, AdvancedEMSynapseMappingOptions, EMSynapseMappingInputNamedTuple
from obi_one.scientific.tasks.em_synapse_mapping.task import EMSynapseMappingTask
from obi_one.scientific.from_id import cell_morphology_from_id, memodel_from_id
from obi_one.core.info import Info

INCLUDE_SPINY_MORPH = True

m = cell_morphology_from_id.CellMorphologyFromID(id_str=str(skeletonized_morphology.id))
m = memodel_from_id.MEModelFromID(id_str=str(skeletonized_morphology.id))
task_config = EMSynapseMappingSingleConfig(
    info=Info(
        campaign_name="EM Synapse Mapping",
        campaign_description="Map EM synapses onto spiny morphology"
    ),
    initialize=EMSynapseMappingSingleConfig.Initialize(
        neurons=EMSynapseMappingInputNamedTuple(name="Skeleton", elements=(m,)),
    ),
    advanced_options=AdvancedEMSynapseMappingOptions(include_spiny_morphologies=INCLUDE_SPINY_MORPH),
    coordinate_output_root="obi-one-test",
)

task = EMSynapseMappingTask(config=task_config)

In [ ]:
task.execute(db_client=entity_client)